# ETL básico con `pygrametl`

## Propósito

Implementar un flujo de extracción, transformación y carga (ETL) que integra ventas de una librería con un catálogo de comunas de Chile. Al finalizar, se verificará que el esquema estrella conserva la cantidad de ejemplares obtenida desde el origen.

## Preparación

- Selecciona un kernel de **Python 3.10**. Instala `pygrametl` y MySQL Connector/Python en el entorno de ese kernel. Si faltan las dependencias, puedes ejecutar `%pip install pygrametl==2.8 mysql-connector-python` en una celda.
- Prepara la base de datos operacional `libreria` con las tablas y los datos usados en el ejemplo. Crea el esquema `ventas_dm` con las tablas `libro`, `tiempo`, `localizacion` y `ventas`, usando **InnoDB** para que la carga pueda revertirse si ocurre un error.
- Guarda `cut_2010_v02.csv` en la carpeta `data`, relativa al directorio de trabajo del notebook. El archivo debe estar en UTF-8, usar comas como separadores e incluir los encabezados `Nombre Comuna` y `Nombre Región`.
- Usa una cuenta MySQL con acceso de lectura a `libreria` y permisos de lectura e inserción en `ventas_dm`. El notebook solicita la contraseña al ejecutar la carga.
- Cada carga completa parte con **las cuatro tablas de destino vacías**. Si necesitas repetir una carga exitosa, vuelve a preparar el esquema de práctica vacío con el script de creación del curso.

## Modelo dimensional

La base operacional `libreria` contiene los pedidos de libros y las comunas de los clientes. El [catálogo territorial de SUBDERE](https://www.subdere.gov.cl/documentacion/regiones-provincias-y-comunas-de-chile-2011) aporta una región para cada comuna del CSV.

**Granularidad de `ventas`.** Cada fila representa el total de ejemplares vendidos de un libro a clientes de una comuna durante un día.

| Tabla | Clave en el almacén | Identificación o contenido |
| --- | --- | --- |
| `libro` | `id_libro` | El par `titulo`, `genero` identifica un libro en este ejemplo. |
| `tiempo` | `id_tiempo` | `fecha` identifica el día. Se conservan también `dia`, `mes`, `anio` y `trimestre`. |
| `localizacion` | `id_localizacion` | `comuna` identifica el miembro. `region` proviene del catálogo territorial. |
| `ventas` | `id_libro`, `id_localizacion`, `id_tiempo` | Las tres claves foráneas identifican el hecho. `cantidad` registra los ejemplares vendidos. |

El ejemplo supone que los nombres de comuna son únicos en el catálogo y coinciden entre las dos fuentes. En un catálogo donde el nombre sea ambiguo, la búsqueda debe usar un código de comuna compartido por ambas fuentes. El mismo criterio se aplica al identificador de libro si título y género resultan insuficientes.

In [ ]:
from pathlib import Path

from IPython.display import Image, display


ruta_diagrama = Path("img/erd-ventas_dm.png")
if ruta_diagrama.is_file():
    display(Image(filename=str(ruta_diagrama), height=450))

## 1. Extracción

`CSVSource` lee cada fila del CSV como un diccionario. Los nombres de las claves provienen de los encabezados y los valores se leen como texto. El flujo abre el archivo durante la precarga de comunas y lo cierra al terminar.

`SQLSource` entrega también un diccionario por fila. Las claves corresponden a los alias de la consulta. La consulta aplica una primera transformación en el origen con `SUM(pedido.cantidad)`, que agrega los ejemplares vendidos al nivel de día, título, género y comuna. Las expresiones completas de `GROUP BY` hacen explícita esa granularidad.

In [ ]:
from contextlib import closing
from getpass import getpass

from mysql.connector import Error, connect, errorcode
from pygrametl import ConnectionWrapper
from pygrametl.datasources import CSVSource, SQLSource
from pygrametl.tables import CachedDimension, FactTable


def conexion(host: str, dbname: str, user: str, pwd: str):
    """Establece una conexión con una base de datos MySQL.

    Args:
        host: Nombre de host o dirección IP del servidor MySQL.
        dbname: Nombre de la base de datos.
        user: Nombre de usuario de MySQL.
        pwd: Contraseña del usuario.

    Returns:
        Una conexión abierta si se logra establecer. Si el conector
        genera un error, muestra un mensaje y devuelve None.
    """
    try:
        return connect(
            host=host, database=dbname, user=user, password=pwd,
            autocommit=False, buffered=True,
        )
    except Error as error:
        if error.errno == errorcode.ER_ACCESS_DENIED_ERROR:
            print(f"Acceso denegado a {dbname}. Revisa el usuario y la contraseña.")
        elif error.errno == errorcode.ER_BAD_DB_ERROR:
            print(f"La base de datos {dbname} no existe.")
        else:
            print(f"No fue posible conectar con {dbname}: {error}")
        return None

In [ ]:
ruta_comunas = Path("data/cut_2010_v02.csv")

consulta_ventas = """
    SELECT
        DATE(pedido.fecha) AS fecha,
        libro.titulo AS titulo,
        genero.nombre AS genero,
        comuna.nombre AS comuna,
        SUM(pedido.cantidad) AS cantidad
    FROM pedido
    INNER JOIN libro ON libro.id_libro = pedido.libro
    INNER JOIN genero ON genero.id_genero = libro.genero
    INNER JOIN cliente ON cliente.id_cliente = pedido.cliente
    INNER JOIN comuna ON comuna.id_comuna = cliente.comuna
    GROUP BY DATE(pedido.fecha), libro.titulo, genero.nombre, comuna.nombre
"""

## 2. Transformación

A partir de la fecha diaria que devuelve MySQL, se derivan el día, el mes, el año y el trimestre. Las funciones modifican directamente el diccionario recibido.

En la carga, `ensure()` obtiene las claves de libro y tiempo y crea los miembros que todavía no existen. `lookup()` busca la clave de la comuna en la dimensión precargada. Con la configuración de este ejemplo, devuelve `None` si la comuna no se encuentra y el proceso revierte la carga.

In [ ]:
def agregar_componentes_fecha(row):
    """Agrega día, mes y año al diccionario de una venta."""
    fecha = row["fecha"]
    row["dia"] = fecha.day
    row["mes"] = fecha.month
    row["anio"] = fecha.year


def agregar_trimestre(row):
    """Agrega el trimestre a partir del mes que contiene el diccionario."""
    row["trimestre"] = (row["mes"] - 1) // 3 + 1

## 3. Carga del almacén

`ConnectionWrapper` adapta la conexión de destino a las abstracciones de `pygrametl`. Cada `CachedDimension` declara la tabla física, su clave y los atributos que se usan para identificar un miembro. La caché permite reutilizar resultados de búsquedas. `FactTable` declara las claves foráneas y la medida que se inserta.

El flujo precarga las comunas, recorre las ventas agregadas y busca las claves de las dimensiones para construir cada hecho. Antes de confirmar la transacción, compara el número de filas y la suma de ejemplares con los de la extracción. También recupera un hecho mediante un `JOIN` para revisar sus atributos y su cantidad. Si ocurre un error, `rollback()` revierte las inserciones en el destino.

In [ ]:
consulta_destino_vacio = """
    SELECT EXISTS (SELECT 1 FROM libro)
        OR EXISTS (SELECT 1 FROM tiempo)
        OR EXISTS (SELECT 1 FROM localizacion)
        OR EXISTS (SELECT 1 FROM ventas) AS hay_datos
"""

consulta_totales = """
    SELECT COUNT(*) AS filas, COALESCE(SUM(cantidad), 0) AS ejemplares
    FROM ventas
"""

consulta_traza = """
    SELECT t.fecha, l.titulo, l.genero, u.comuna, u.region, v.cantidad
    FROM ventas AS v
    JOIN tiempo AS t ON t.id_tiempo = v.id_tiempo
    JOIN libro AS l ON l.id_libro = v.id_libro
    JOIN localizacion AS u ON u.id_localizacion = v.id_localizacion
    WHERE v.id_libro = %(id_libro)s
      AND v.id_localizacion = %(id_localizacion)s
      AND v.id_tiempo = %(id_tiempo)s
"""

In [ ]:
def cargar_ventas(host, usuario, contrasena, archivo_comunas):
    """Carga el esquema estrella y devuelve los totales y la traza de una fila."""
    cnx_libreria = conexion(host, "libreria", usuario, contrasena)
    if cnx_libreria is None:
        raise RuntimeError("Se requiere una conexión al origen libreria.")

    with closing(cnx_libreria):
        cnx_dm = conexion(host, "ventas_dm", usuario, contrasena)
        if cnx_dm is None:
            raise RuntimeError("Se requiere una conexión al destino ventas_dm.")

        with closing(cnx_dm):
            try:
                cnx_dm_wrapper = ConnectionWrapper(connection=cnx_dm)

                with closing(cnx_dm.cursor(dictionary=True)) as cursor:
                    cursor.execute(consulta_destino_vacio)
                    if cursor.fetchone()["hay_datos"]:
                        raise ValueError("La carga requiere las cuatro tablas de destino vacías.")

                libro_dim = CachedDimension(
                    name="libro", key="id_libro",
                    attributes=["titulo", "genero"], lookupatts=["titulo", "genero"],
                    targetconnection=cnx_dm_wrapper,
                )
                tiempo_dim = CachedDimension(
                    name="tiempo", key="id_tiempo",
                    attributes=["fecha", "dia", "mes", "anio", "trimestre"],
                    lookupatts=["fecha"], targetconnection=cnx_dm_wrapper,
                )
                localizacion_dim = CachedDimension(
                    name="localizacion", key="id_localizacion",
                    attributes=["comuna", "region"], lookupatts=["comuna"],
                    targetconnection=cnx_dm_wrapper,
                )
                ventas_ft = FactTable(
                    name="ventas", keyrefs=["id_libro", "id_localizacion", "id_tiempo"],
                    measures=["cantidad"], targetconnection=cnx_dm_wrapper,
                )

                # Precarga de la dimensión territorial desde el CSV.
                comunas = set()
                with open(archivo_comunas, encoding="utf-8-sig", newline="") as archivo:
                    for row in CSVSource(f=archivo, delimiter=","):
                        comuna = row["Nombre Comuna"]
                        region = row["Nombre Región"]
                        if not comuna or not region:
                            raise ValueError("El CSV contiene una comuna o una región vacía.")
                        if comuna.casefold() in comunas:
                            raise ValueError(f"Comuna duplicada en el CSV: {comuna!r}.")
                        comunas.add(comuna.casefold())
                        localizacion_dim.insert({"comuna": comuna, "region": region})

                # El lote se transforma y se carga fila por fila.
                filas_origen = 0
                ejemplares_origen = 0
                traza = {}
                for row in SQLSource(connection=cnx_libreria, query=consulta_ventas):
                    if filas_origen == 0:
                        traza["Extraída"] = row.copy()

                    agregar_componentes_fecha(row)
                    agregar_trimestre(row)
                    row["id_localizacion"] = localizacion_dim.lookup(row)
                    if row["id_localizacion"] is None:
                        raise ValueError(f"La comuna {row['comuna']!r} no está en el catálogo.")

                    row["id_libro"] = libro_dim.ensure(row)
                    row["id_tiempo"] = tiempo_dim.ensure(row)
                    hecho = {
                        "id_libro": row["id_libro"],
                        "id_localizacion": row["id_localizacion"],
                        "id_tiempo": row["id_tiempo"],
                        "cantidad": row["cantidad"],
                    }
                    ventas_ft.insert(hecho)

                    if filas_origen == 0:
                        traza["Transformada"] = row.copy()
                        traza["Cargada (claves y medida)"] = hecho
                    filas_origen += 1
                    ejemplares_origen += row["cantidad"]

                # Controles antes de confirmar las inserciones.
                with closing(cnx_dm.cursor(dictionary=True)) as cursor:
                    cursor.execute(consulta_totales)
                    totales = cursor.fetchone()
                    if (totales["filas"], totales["ejemplares"]) != (
                        filas_origen, ejemplares_origen
                    ):
                        raise ValueError("Los totales de origen y destino difieren.")

                    if filas_origen:
                        cursor.execute(consulta_traza, traza["Cargada (claves y medida)"])
                        fila_consultada = cursor.fetchone()
                        if fila_consultada is None or any(
                            fila_consultada[campo] != valor
                            for campo, valor in traza["Extraída"].items()
                        ):
                            raise ValueError("El hecho consultado difiere de la fila extraída.")
                        traza["Consultada con JOIN"] = fila_consultada

                cnx_dm_wrapper.commit()
            except BaseException:
                cnx_dm.rollback()
                raise

    return totales, traza

## 4. Ejecución y lectura de resultados

La siguiente celda solicita las credenciales y ejecuta el flujo completo. La última celda muestra los totales verificados y el recorrido de la primera fila del resultado SQL. Si la consulta de origen devuelve cero filas, se informan los totales y la traza queda vacía.

In [ ]:
host = "localhost"
usuario = input("Usuario MySQL: ")
contrasena = getpass("Contraseña MySQL: ")
try:
    totales, traza = cargar_ventas(host, usuario, contrasena, ruta_comunas)
finally:
    del contrasena

In [ ]:
print(f"Filas de hechos: {totales['filas']}")
print(f"Ejemplares vendidos: {totales['ejemplares']}")

for etapa, fila in traza.items():
    print(etapa)
    display(fila)

### Para discutir

Considera dos pedidos del mismo libro para clientes de la misma comuna, efectuados a horas distintas del mismo día. ¿Cuántas filas se extraen y qué cantidad almacena el hecho? ¿Qué cambia si uno de los clientes pertenece a otra comuna?